# Base escolhida: Medical Appointment No Shows.csv

## Questão 1

**Cenário:** Classificação Binária Supervisionada.

**Justificativa:** O objetivo central é prever se um paciente irá comparecer ou não à consulta, utilizando o atributo alvo `No-show`. 

* **Classificação:** A variável dependente é categórica (discreta) e não um valor numérico contínuo.
* **Supervisionada:** O modelo aprenderá a partir de dados históricos que já possuem os rótulos (resultados) conhecidos.
* **Binária:** Como existem apenas duas classes possíveis na base ("Yes" ou "No"), o problema é definido especificamente como classificação binária.

In [3]:
import pandas as pd
import numpy as np

In [16]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

path = 'Data/Medical Appointment No Shows.csv'
df = pd.read_csv(path)

df.head()

,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
0,2.987250e+13,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No
1,5.589978e+14,5642503,M,2016-04-29T16:08:27Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No
2,4.262962e+12,5642549,F,2016-04-29T16:19:04Z,2016-04-29T00:00:00Z,62,MATA DA PRAIA,0,0,0,0,0,0,No
3,8.679512e+11,5642828,F,2016-04-29T17:29:31Z,2016-04-29T00:00:00Z,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No
4,8.841186e+12,5642494,F,2016-04-29T16:07:23Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,1,1,0,0,0,No


## Código reutilizavel de diagnostico
#### diagnostico_inicial(data)

In [17]:
path = 'Data/Medical Appointment No Shows.csv'
df = pd.read_csv(path)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

def diagnostico_inicial(data):
    # Dimensão da base de dados
    n_linhas, n_colunas = data.shape
    print(f'Número de linhas: {n_linhas}\nNúmero de colunas: {n_colunas}\n')
    
    # Tipos de dados e valores ausentes
    info_colunas = pd.DataFrame({
        'Tipo': data.dtypes,
        'Ausentes': data.isnull().sum(),
        'Cardinalidade': data.nunique()
    })
    print(f'{info_colunas}\n')
    
    # Duplicidade
    total_duplicados = data.duplicated().sum()
    print(f'Número de linhas duplicadas: {total_duplicados}\n')
    
    # Inconsistências
    print("Inconsistências\n")
    
    if 'Age' in data.columns:
    # Idades negativas
        idades_negativas = (data['Age'] < 0).sum()
        print(f"Linhas com idade negativa: {idades_negativas}")
    # Idades muito grandes
        idades_grandes = (data['Age'] > 110).sum()
        print(f"Linhas com idade maior que 110: {idades_grandes}\n")
    
    # ver se consulta é antes do agendado (pode ou não fazer sentido)
    if 'ScheduledDay' in data.columns and 'AppointmentDay' in data.columns:
        schedule_day = pd.to_datetime(data['ScheduledDay']).dt.date
        appointment_day = pd.to_datetime(data['AppointmentDay']).dt.date
        print(f"Consultas retroativas: {(appointment_day < schedule_day).sum()}")
        
    # Handcap: nos podemos analisar handcap de duas formas, vendo como binario (tem ou não tem deficiencia), ou como numero de deficiencias (0, 1, 2, 3, 4)
    if 'Handcap' in data.columns:
        contagem = data['Handcap'].value_counts().sort_index()
        for valor, total in contagem.items():
            print(f"Handcap {valor}: {total} pacientes")
            
            
    # Verificação Automática de Categorias Raras (< 1% da base) 
    print("Alerta de Categorias Raras (< 1% da base)")
    limite_raro = 0.01 * n_linhas
    for col in data.select_dtypes(include=['object', 'int64']).columns:
        if data[col].nunique() < 100 and col not in ['PatientId', 'AppointmentID']:
            counts = data[col].value_counts()
            raras = counts[counts < limite_raro]
            if not raras.empty:
                print(f"Coluna '{col}': {len(raras)} categoria(s) com baixíssima frequência.")
                for cat, val in raras.items():
                    print(f"  > Categoria '{cat}': {val} ocorrências")
    

diagnostico_inicial(df)

Número de linhas: 110527
Número de colunas: 14

                   Tipo  Ausentes  Cardinalidade
PatientId       float64         0          62299
AppointmentID     int64         0         110527
Gender              str         0              2
ScheduledDay        str         0         103549
AppointmentDay      str         0             27
Age               int64         0            104
Neighbourhood       str         0             81
Scholarship       int64         0              2
Hipertension      int64         0              2
Diabetes          int64         0              2
Alcoholism        int64         0              2
Handcap           int64         0              5
SMS_received      int64         0              2
No-show             str         0              2

Número de linhas duplicadas: 0

Inconsistências

Linhas com idade negativa: 1
Linhas com idade maior que 110: 5

Consultas retroativas: 5
Handcap 0: 108286 pacientes
Handcap 1: 2042 pacientes
Handcap 2: 183 pacientes

/var/folders/jn/w8qgbc71103fy_919c_3kj7c0000gn/T/ipykernel_74574/1161460308.py:53: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in data.select_dtypes(include=['object', 'int64']).columns:
